# IndoBERT — Balanced 3000 (label Opus + rubrik domain-aware)

Fine-tune `indobenchmark/indobert-base-p1` pada subset **balanced 1000/kelas** dengan label
TERBARU (re-label Claude Opus + rubrik domain-aware). **Tanpa upload manual** — notebook
meng-clone repo via **GitHub PAT** dan baca data dari **MongoDB** (`processed_bert` sudah
ber-label baru). Split kanonik identik SVM (urut comment_id, seed=42, 70/20/10 -> 2100/600/300).

**Sebelum Run all:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Simpan 2 Colab Secret (ikon kunci 🔑 di kiri): **`GITHUB_PAT`** dan **`MONGO_URI`** (toggle *Notebook access* ON). Kalau tak ada, akan diminta ketik manual.
3. Run all. Di akhir, `indobert_balanced3k_metrics.json` otomatis ter-download — kirim ke Claude.

In [ ]:
!nvidia-smi -L || echo 'TIDAK ADA GPU — Runtime > Change runtime type > T4 GPU'

In [ ]:
# --- Ambil GITHUB_PAT & MONGO_URI (Colab Secrets, fallback ketik manual) ---
import os
def _secret(name):
    v = os.environ.get(name, '')
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(name) or ''
        except Exception:
            v = ''
    if not v:
        from getpass import getpass
        v = getpass(f'{name}: ')
    return v

GITHUB_PAT = _secret('GITHUB_PAT')
os.environ['MONGO_URI'] = _secret('MONGO_URI')
print('PAT:', bool(GITHUB_PAT), '| MONGO_URI:', bool(os.environ['MONGO_URI']))

In [ ]:
# --- Clone repo via PAT (token tidak ter-echo) ---
import subprocess, shutil, pathlib
REPO_DIR = '/content/jokowi_sentiment_project'
if pathlib.Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
url = f'https://{GITHUB_PAT}@github.com/Arachnoida/jokowi_sentiment_project.git'
r = subprocess.run(['git', 'clone', '--depth', '1', url, REPO_DIR],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('clone OK' if r.returncode == 0 else 'CLONE GAGAL (cek PAT/akses repo):')
print('\n'.join(l for l in r.stdout.splitlines() if 'github.com' not in l))
assert r.returncode == 0
assert pathlib.Path(REPO_DIR, 'outputs/labeling/balanced_3000.csv').exists(), 'balanced_3000.csv tak ada di repo — sudah ter-push?'

In [ ]:
# --- Dependensi (torch+transformers biasanya sudah ada di Colab) ---
!pip -q install -r {REPO_DIR}/requirements-colab.txt pymongo[srv] certifi python-dotenv

In [ ]:
# --- Latih IndoBERT pakai trainer kanonik repo (MONGO_URI diwariskan dari os.environ) ---
# processed_bert di Mongo SUDAH ber-label baru; subset = balanced_3000.csv dari repo.
!cd {REPO_DIR} && python -m src.modeling.train_indobert --subset outputs/labeling/balanced_3000.csv --tag balanced3k

In [ ]:
# --- Ringkasan + download artefak ---
import json
rep = f'{REPO_DIR}/outputs/reports'
m = json.load(open(f'{rep}/indobert_balanced3k_metrics.json'))['test']
print('IndoBERT balanced3k  acc=%.4f  macro-F1=%.4f' % (m['accuracy'], m['macro_f1']))
for l in ['Negatif','Netral','Positif']:
    pc = m['per_class'][l]; print('  %-8s P=%.3f R=%.3f F1=%.3f' % (l, pc['precision'], pc['recall'], pc['f1-score']))
try:
    from google.colab import files
    for f in ['indobert_balanced3k_metrics.json','indobert_balanced3k_test_confusion.png','indobert_balanced3k_predictions.csv']:
        p = f'{rep}/{f}'
        if pathlib.Path(p).exists():
            files.download(p)
except Exception as ex:
    print('Download manual dari panel Files:', rep, '| detail:', ex)